# Synthetic data generation (V2)

This notebook generates a synthetic student-enrolment dataset that mirrors the real
`student_aggregate` schema (one fact table plus its dimension tables). The design comprises three
deliberate layers of "realism":

1. **V1 weighted categoricals** - simple fields (gender, age band, course level, ...) are drawn
   from real category proportions supplied in `distribution_values.csv`.
2. **V2 conditional distributions** - fields that depend on another field (EFTSL, credit points,
   commencing/continuing) are drawn from industry-partner distributions conditioned on broad
   field of education and/or teaching period.
3. **In-house Beta distributions** - the attrition flags are drawn from Beta parameters supplied
   by the in-house team.

Every distribution lives in a curated CSV in the project's `curated/` folder, so the statistics
can be updated without touching code. Nothing is hard-coded and no values are invented: a column
is populated only if it is drawn from a supplied distribution or deterministically derived from
one. Any schema column that cannot be grounded is left NULL.

## Setup

In [0]:
# dbldatagen (Databricks Labs Data Generator) is the library that builds the base synthetic rows.
%pip install dbldatagen

In [0]:
# Core imports: dbldatagen for row generation, and pyspark functions (aliased F) for all column logic.
import dbldatagen as dg
import pyspark.sql.functions as F


# align_to_target is the single most important helper in this notebook. It takes a constructed
# DataFrame (df_in) and reshapes it to EXACTLY match a real target table's schema, so the result can be
# written straight to that Delta table.
#
# How it works: the target table is read only to borrow its schema (column names, order, and types).
# Then, for each column the target expects:
#   - if col_map supplies an expression for it, that expression is used, cast to the target
#     column's declared type;
#   - otherwise a typed NULL is emitted.
# This guarantees the output always has the right columns in the right order with the right types, and
# makes the deliberate absence of a value for a column explicit (it becomes a typed NULL) rather than
# causing a schema mismatch.
#
# type_overrides: optional dict of column_name -> Spark SQL type string. Used when the existing Delta
# table declares a type that would truncate generated values (e.g. eftsl as decimal(10,0) drops
# fractional enrolment load). overwriteSchema on write persists the corrected type.
def align_to_target(df_in, target_fqn, col_map, type_overrides=None):
    type_overrides = type_overrides or {}
    template = spark.table(target_fqn)  # borrow the target schema only; no data is read into memory
    parts = []
    for field in template.schema.fields:
        dt = field.dataType
        if field.name in col_map:
            out_type = type_overrides.get(field.name, dt)
            parts.append(col_map[field.name].cast(out_type).alias(field.name))
        else:
            # No grounded value for this column: write a typed NULL rather than invent data.
            parts.append(F.lit(None).cast(dt).alias(field.name))
    return df_in.select(parts)


## V1 config-driven weighted categoricals

The simplest distributions: a set of categorical fields (gender, age band, course level, ...)
where each possible value has a real-world weight (relative frequency). The values and
weights are kept in `distribution_values.csv`, and a set of unweighted lookup lists in `lookup_values.csv`,
then fed to dbldatagen. Storing them in CSV means the proportions can be re-tuned without
editing this notebook.

In [0]:
# The curated CSVs live in this project's Workspace folder. Spark reads Workspace files via the
# "file:" scheme, so every path is prefixed with it. Changing WORKSPACE_PROJECT_PATH re-points the
# whole notebook at a different deployment.
WORKSPACE_PROJECT_PATH = "/Workspace/Shared/Shared folder/Data Generation V2"
CURATED_BASE_PATH = f"{WORKSPACE_PROJECT_PATH}/curated"

DISTRIBUTION_VALUES_PATH = f"file:{CURATED_BASE_PATH}/distribution_values.csv"
LOOKUP_VALUES_PATH = f"file:{CURATED_BASE_PATH}/lookup_values.csv"


# Single helper for reading any curated CSV. Every config file in this notebook (V1, V2 and the Beta
# parameters) is read through this one function so the loading behaviour is identical everywhere.
# inferSchema is off so every value arrives as a raw string; each value is cast explicitly (see
# cast_config_value) to retain full control over types. .collect() pulls the small config table (a few
# hundred rows at most) into the driver as plain Python dicts.
def _read_config_rows(path):
    sdf = spark.read.option("header", "true").option("inferSchema", "false").csv(path)
    return [row.asDict() for row in sdf.collect()]


# The CSVs carry a data_type column so each value can be turned back into the right Python type.
# (Everything is read as text, so e.g. "2021" must become int 2021, "true" must become boolean True.)
def cast_config_value(value, data_type):
    if value is None:
        return None
    dt = (data_type or "string").strip().lower()
    if dt in ("int", "integer", "bigint", "long"):
        return int(float(value))  # via float first so "2021.0" style strings still parse
    if dt in ("double", "float", "decimal"):
        return float(value)
    if dt in ("bool", "boolean"):
        return str(value).strip().lower() in ("true", "1", "yes")
    return str(value)


# Load the two V1 config files once into driver memory, so repeated lookups below are cheap.
_DISTRIBUTION_ROWS = _read_config_rows(DISTRIBUTION_VALUES_PATH)
_LOOKUP_ROWS = _read_config_rows(LOOKUP_VALUES_PATH)


# For a weighted feature, return two parallel lists dbldatagen expects: the possible values and their
# integer weights. Example: gender -> (["Female", "Male", ...], [58, 41, ...]). dbldatagen then picks
# values in proportion to the weights.
def get_weighted_distribution(feature_name):
    rows = [r for r in _DISTRIBUTION_ROWS if r["feature_name"] == feature_name]
    if not rows:
        raise ValueError(f"No distribution values found for feature '{feature_name}'")
    values = [cast_config_value(r["value"], r.get("data_type")) for r in rows]
    weights = [int(float(r["weight"])) for r in rows]
    return values, weights


# For an unweighted lookup feature, return just the list of allowed values (used later to cycle
# through field-of-education labels when building dimension rows).
def get_lookup_values(feature_name):
    rows = [r for r in _LOOKUP_ROWS if r["feature_name"] == feature_name]
    if not rows:
        raise ValueError(f"No lookup values found for feature '{feature_name}'")
    return [cast_config_value(r["value"], r.get("data_type")) for r in rows]


# Convenience wrapper: attach one weighted categorical column to a dbldatagen spec straight from the
# config CSV. random=True makes each row an independent weighted draw.
def add_weighted_column(spec, output_column, feature_name, dbldatagen_type):
    values, weights = get_weighted_distribution(feature_name)
    return spec.withColumn(
        output_column,
        dbldatagen_type,
        values=values,
        weights=weights,
        random=True,
    )

## V2 conditional distributions

These fields are not independent - their distribution *depends on* another field. For example,
the spread of enrolment load (EFTSL) differs by field of education and teaching period, and
credit-point outcomes differ by field of education. The industry partner supplied these as
counts per (condition, outcome) combination.

Each supplied table is turned into a **weighted index** (condition -> outcomes + weights) and is
then sampled per student with a small user-defined function (UDF). The sampling is made
**deterministic** (seeded from the row id) so the same row always yields the same value - re-runs
are reproducible, which matters for a dataset that is regenerated repeatedly.

In [0]:
# Curated V2 config file paths. Each file holds counts for a conditional distribution; the band file
# translates band labels (e.g. "0-0.125") into numeric ranges.
V2_EFTSL_PATH = f"file:{CURATED_BASE_PATH}/eftsl_by_foe_teaching_period.csv"
V2_CC_FOE_PATH = f"file:{CURATED_BASE_PATH}/commencing_continuing_by_foe.csv"
V2_CC_TP_PATH = f"file:{CURATED_BASE_PATH}/commencing_continuing_by_teaching_period.csv"
V2_CP_PASSED_PATH = f"file:{CURATED_BASE_PATH}/credit_points_passed_by_foe.csv"
V2_CP_ENROLLED_PATH = f"file:{CURATED_BASE_PATH}/credit_points_enrolled_by_foe.csv"
V2_CP_FAILED_PATH = f"file:{CURATED_BASE_PATH}/credit_points_failed_by_foe.csv"
V2_CP_WITHDRAWN_PATH = f"file:{CURATED_BASE_PATH}/credit_points_withdrawn_by_foe.csv"
V2_BANDS_PATH = f"file:{CURATED_BASE_PATH}/band_definitions.csv"


# Turn a flat list of config rows into a fast lookup structure for weighted sampling.
#
# Input rows look like: {condition column(s), an outcome value, a weight/count}.
# The rows are grouped by the condition column(s) and, for each group, a CUMULATIVE weight list is
# precomputed. That cumulative list is what enables one-step sampling later: draw a number between 0 and
# the total, then find which outcome's cumulative band it falls into (the "roulette wheel" / inverse-CDF trick).
#
# Result shape:  index[(condition values)] = (outcomes, cumulative_weights, total_weight)
# Example:       index[("HEALTH",)] = (["0-72", "72-144", ...], [500, 900, ...], 1200)
def _build_weighted_index(rows, cond_cols, value_col, weight_col):
    from collections import defaultdict

    grouped = defaultdict(list)
    for r in rows:
        weight = float(r[weight_col])
        if weight <= 0:
            continue  # zero/negative counts carry no probability mass, so skip them
        key = tuple(r[c] for c in cond_cols)
        grouped[key].append((r[value_col], weight))

    index = {}
    for key, pairs in grouped.items():
        values = [p[0] for p in pairs]
        cumulative = []
        running = 0.0
        for _, w in pairs:
            running += w
            cumulative.append(running)  # e.g. weights [500,400,300] -> cumulative [500,900,1200]
        index[key] = (values, cumulative, running)
    return index


# Band definitions: many V2 outcomes are given as bands (e.g. EFTSL "0-0.125", credit points "72-144").
# This builds a lookup band_defs[band_type][band_label] = (lower_bound, upper_bound) so a chosen band
# can later be turned into an actual number by sampling a point inside the range.
_band_rows = _read_config_rows(V2_BANDS_PATH)
BAND_DEFS = {}
for _r in _band_rows:
    BAND_DEFS.setdefault(_r["band_type"], {})[_r["band_label"]] = (
        float(_r["lower_bound"]),
        float(_r["upper_bound"]),
    )

# EFTSL (enrolment load) band distribution. Built at two levels of detail:
#   EFTSL_IDX      - conditioned on BOTH field of education and teaching period (the precise version)
#   EFTSL_IDX_FOE  - conditioned on field of education only (a coarser fallback used when a specific
#                    field+period combination isn't present in the data)
_eftsl_rows = _read_config_rows(V2_EFTSL_PATH)
EFTSL_IDX = _build_weighted_index(_eftsl_rows, ["field_of_education", "teaching_period"], "eftsl_band", "enrolment_count")
EFTSL_IDX_FOE = _build_weighted_index(_eftsl_rows, ["field_of_education"], "eftsl_band", "enrolment_count")

# Commencing vs continuing student stage. Two views: by field of education (used at the row's
# calendar-year grain) and by teaching period (a period-grain variant).
CC_FOE_IDX = _build_weighted_index(_read_config_rows(V2_CC_FOE_PATH), ["field_of_education"], "student_stage", "student_count")
CC_TP_IDX = _build_weighted_index(_read_config_rows(V2_CC_TP_PATH), ["teaching_period"], "student_stage", "student_count")

# Cumulative credit-point bands, conditioned on field of education. One index per outcome measure
# (passed / enrolled / failed / withdrawn) because each has its own distribution.
CP_PASSED_IDX = _build_weighted_index(_read_config_rows(V2_CP_PASSED_PATH), ["field_of_education"], "credit_point_band", "student_count")
CP_ENROLLED_IDX = _build_weighted_index(_read_config_rows(V2_CP_ENROLLED_PATH), ["field_of_education"], "credit_point_band", "student_count")
CP_FAILED_IDX = _build_weighted_index(_read_config_rows(V2_CP_FAILED_PATH), ["field_of_education"], "credit_point_band", "student_count")
CP_WITHDRAWN_IDX = _build_weighted_index(_read_config_rows(V2_CP_WITHDRAWN_PATH), ["field_of_education"], "credit_point_band", "student_count")

# Registry of all indexes, keyed by a short name. The sampling UDFs below look their index up by key.
# IMPORTANT (serverless compatibility): these are plain Python dicts. On serverless / Spark Connect
# sparkContext.broadcast is unavailable, so each UDF instead CAPTURES the dict it needs via a closure and
# Spark ships it to the workers automatically. The dicts are tiny (a few hundred entries), so this is
# cheap.
_BC = {
    "eftsl": EFTSL_IDX,
    "eftsl_foe": EFTSL_IDX_FOE,
    "cc_foe": CC_FOE_IDX,
    "cc_tp": CC_TP_IDX,
    "cp_passed": CP_PASSED_IDX,
    "cp_enrolled": CP_ENROLLED_IDX,
    "cp_failed": CP_FAILED_IDX,
    "cp_withdrawn": CP_WITHDRAWN_IDX,
    "bands": BAND_DEFS,
}

from pyspark.sql.types import StringType, DoubleType, IntegerType


# Factory that builds a UDF to PICK A CATEGORY conditioned on one or more columns.
#
# "Factory" means it returns a UDF: it is called once (choosing which index and a salt) and hands
# back a function Spark applies per row. bc_key selects which weighted index to sample from; salt is a
# label mixed into the hash so different columns built from the same index still get independent draws.
#
# Determinism: instead of a random number generator, the function hashes (row_id + salt + condition)
# into a number in [0, 1). Same inputs -> same hash -> same choice on every run. That value is scaled to
# the group's total weight and bisect (binary search) over the cumulative weights locates which category
# it lands on - the inverse-CDF / roulette-wheel selection described above. Returns None if the condition is unseen.
def make_categorical_udf(bc_key, salt):
    index = _BC[bc_key]  # captured by closure and shipped to workers (no broadcast on serverless)

    @F.udf(StringType())
    def _pick(row_id, *cond):
        import bisect
        import hashlib

        entry = index.get(tuple(cond))
        if entry is None:
            return None  # no distribution for this condition -> leave the value NULL
        values, cumulative, total = entry
        # Hash -> a stable pseudo-random fraction u in [0, 1) for this specific row + condition.
        digest = hashlib.md5(f"{row_id}|{salt}|{'|'.join(str(c) for c in cond)}".encode("utf-8")).hexdigest()
        u = (int(digest[:12], 16) % 1_000_000_000) / 1_000_000_000.0
        target = u * total  # a point on the cumulative-weight axis
        i = bisect.bisect_left(cumulative, target)  # which category's band contains it
        if i >= len(values):
            i = len(values) - 1
        return values[i]

    return _pick


# Factory that builds a UDF to PICK A BAND (as above) and then RETURN A NUMBER inside that band.
#
# Two-step draw: (1) choose a band using the same weighted / inverse-CDF logic as the categorical UDF,
# then (2) pick a value uniformly within the band's [lower, upper) range. Two independent hash draws are
# used ("band" and "within"), tagged so they don't correlate.
# Extras:
#   fallback_bc_key - if the precise condition (e.g. field+period) is missing, fall back to a coarser
#                     index keyed on just the first condition (e.g. field only).
#   default_lo/hi   - last-resort range if even the fallback has nothing.
#   as_int          - round to a whole number (credit points) vs keep a decimal (EFTSL).
def make_banded_numeric_udf(bc_key, band_type, salt, as_int, fallback_bc_key=None, default_lo=0.0, default_hi=0.0):
    index = _BC[bc_key]
    fallback_index = _BC[fallback_bc_key] if fallback_bc_key else None
    band_defs = _BC["bands"]
    return_type = IntegerType() if as_int else DoubleType()

    @F.udf(return_type)
    def _sample(row_id, *cond):
        import bisect
        import hashlib

        # Local helper: a stable pseudo-random fraction in [0, 1), one per (row, tag, condition).
        def uniform(tag):
            digest = hashlib.md5(f"{row_id}|{salt}|{tag}|{'|'.join(str(c) for c in cond)}".encode("utf-8")).hexdigest()
            return (int(digest[:12], 16) % 1_000_000_000) / 1_000_000_000.0

        entry = index.get(tuple(cond))
        if entry is None and fallback_index is not None:
            entry = fallback_index.get((cond[0],))  # try the coarser (first-condition-only) index

        if entry is None:
            lo, hi = default_lo, default_hi  # nothing matched: use the safe default range
        else:
            # Step 1: choose a band via weighted inverse-CDF selection.
            values, cumulative, total = entry
            target = uniform("band") * total
            i = bisect.bisect_left(cumulative, target)
            if i >= len(values):
                i = len(values) - 1
            lo, hi = band_defs.get(band_type, {}).get(values[i], (default_lo, default_hi))

        # Step 2: pick a concrete value uniformly inside the chosen band.
        sampled = lo + uniform("within") * (hi - lo)
        if as_int:
            return int(round(sampled))
        return float(round(sampled, 4))

    return _sample

# ---- Attrition outcome via in-house Beta distribution parameters ----
#
# Attrition (did a student drop out?) is a yes/no outcome, but HOW LIKELY a student is to drop out
# varies. The in-house team modelled that likelihood with a Beta distribution per field of education.
#
# Two statistical ideas make this work:
#   - BETA DISTRIBUTION: a probability distribution over values between 0 and 1, so it is the natural
#     way to describe "a rate/probability that is itself uncertain". Its shape is set by two positive
#     numbers, alpha and beta; the average rate it produces is alpha / (alpha + beta). Here alpha and
#     beta act like evidence counts (roughly: students who stayed vs students who attritted).
#   - BERNOULLI TRIAL: a single yes/no event with a fixed success probability p - one flip of a
#     weighted coin.
#
# So per student a two-step draw is performed (exactly the distribution creator's method):
#   1. draw an attrition PROBABILITY p from the Beta for the student's field of education, then
#   2. run one Bernoulli trial: attritted = (random number in [0,1)) < p.
# The intermediate probability is never stored as a column - it lives only inside the UDF, and only the
# final boolean flag is written. These parameters are used ONLY for the attrition flags, nothing else.

from pyspark.sql.types import BooleanType

ATTRITION_BETA_PATH = f"file:{CURATED_BASE_PATH}/attrition_beta_parameters.csv"

# The config supplies three non-overlapping commencing-cohort blocks, each from a dataset with a
# different OBSERVATION WINDOW: "2005-2015" (9-year), "2016-2018" (6-year) and "2019-2020" (4-year).
# The window length matters: only the 9-year block can observe long-duration dropouts (a student who
# leaves in year 7 is censored in the 4- and 6-year blocks). The 9-year block is therefore used alone -
# it is methodologically consistent (single window), respects full 9-year dropout durations, has the
# largest per-field sample, and matches the distribution creator's stated preference. Measured rates
# are stable across blocks (~16-18%), so this loses little on recency.
# Multiple blocks may be listed to POOL them (Alpha/Beta are summed as Beta/Binomial evidence counts),
# but pooling mixes observation windows and is dominated by the 9-year counts, so it is not recommended.
BETA_COHORT_PERIODS = ["2005-2015"]

_pooled_beta = {}
for _r in _read_config_rows(ATTRITION_BETA_PATH):
    if _r["Commencing_Years"] not in BETA_COHORT_PERIODS:
        continue
    _foe = _r["Broad_FOE"].strip().upper()
    _a, _b = _pooled_beta.get(_foe, (0.0, 0.0))
    _pooled_beta[_foe] = (_a + float(_r["Alpha"]), _b + float(_r["Beta"]))
if not _pooled_beta:
    raise ValueError(f"No attrition Beta parameters found for periods {BETA_COHORT_PERIODS}")

# field of education (upper-cased to match generated broad_field_of_education) -> pooled (alpha, beta)
ATTRITION_BETA_BY_FOE = _pooled_beta
# Pooled-average fallback for any generated field of education without its own row (e.g. "NONE").
_beta_alphas = [a for a, _ in _pooled_beta.values()]
_beta_betas = [b for _, b in _pooled_beta.values()]
ATTRITION_BETA_FALLBACK = (
    sum(_beta_alphas) / len(_beta_alphas),
    sum(_beta_betas) / len(_beta_betas),
)


def make_attrition_udf(params_by_foe, fallback, salt):
    """Return a UDF that draws a boolean attrition flag from the Beta parameters for a row's field of
    education. Deterministic per row id so runs are reproducible: the Beta draw is seeded from the id,
    and the Bernoulli outcome uses a second independent hash-derived uniform."""

    @F.udf(BooleanType())
    def _draw(row_id, foe):
        import hashlib
        import numpy as np

        # Look up this student's (alpha, beta); use the pooled-average fallback if the field has no row.
        alpha, beta = params_by_foe.get((foe or "").strip().upper(), fallback)
        # Step 1 - draw the attrition probability p from the Beta. Seed the RNG from the row id so the
        # same row always draws the same p (reproducible), yet different rows differ.
        seed = int(hashlib.md5(f"{row_id}|{salt}".encode("utf-8")).hexdigest()[:8], 16)
        prob = float(np.random.default_rng(seed).beta(alpha, beta))
        # Step 2 - the Bernoulli trial: a second, independent stable uniform in [0,1); attrition is true
        # when it falls below p. (Separate "bernoulli" salt so the coin flip doesn't correlate with p.)
        digest = hashlib.md5(f"{row_id}|{salt}|bernoulli".encode("utf-8")).hexdigest()
        uniform = (int(digest[:12], 16) % 1_000_000_000) / 1_000_000_000.0
        return bool(uniform < prob)

    return _draw

# ---- Course admission load category (Full-time / Part-time) via in-house Beta parameters ----
# Partner curated file beta_parameters_by_att_type.csv (Attendance_Type = Full-time / Part-time). Same
# 9-year cohort window as attrition. P(Full-time) uses pooled Beta evidence (alpha + beta) per type.
ATT_TYPE_BETA_PATH = f"file:{CURATED_BASE_PATH}/beta_parameters_by_att_type.csv"

_pooled_att_type = {}
for _r in _read_config_rows(ATT_TYPE_BETA_PATH):
    if _r["Commencing_Years"] not in BETA_COHORT_PERIODS:
        continue
    _att = _r["Attendance_Type"].strip()
    _a, _b = _pooled_att_type.get(_att, (0.0, 0.0))
    _pooled_att_type[_att] = (_a + float(_r["Alpha"]), _b + float(_r["Beta"]))
if "Full-time" not in _pooled_att_type or "Part-time" not in _pooled_att_type:
    raise ValueError(f"No attendance-type Beta parameters found for periods {BETA_COHORT_PERIODS}")

LOAD_CATEGORY_FT_PARAMS = _pooled_att_type["Full-time"]
LOAD_CATEGORY_PT_PARAMS = _pooled_att_type["Part-time"]


def make_load_category_udf(ft_params, pt_params, salt):
    """Draw Full-time vs Part-time from pooled att_type Beta evidence (deterministic per row id)."""

    alpha_ft, beta_ft = ft_params
    alpha_pt, beta_pt = pt_params
    evidence_ft = alpha_ft + beta_ft
    evidence_pt = alpha_pt + beta_pt
    p_full_time = evidence_ft / (evidence_ft + evidence_pt)

    @F.udf(StringType())
    def _draw(row_id):
        import hashlib

        digest = hashlib.md5(f"{row_id}|{salt}|load_category".encode("utf-8")).hexdigest()
        uniform = (int(digest[:12], 16) % 1_000_000_000) / 1_000_000_000.0
        return "Full-time" if uniform < p_full_time else "Part-time"

    return _draw

## Build the enrolment fact rows

This is the core of the notebook. The notebook generates base rows with dbldatagen (the V1
weighted categoricals), then enriches them in four clearly separated LAYERS:

1. **Identity & base attributes** - de-identified hashes, the teaching-period key, dates, simple
   derived flags.
2. **Conditional distributions (V2)** - student stage, EFTSL, credit points, and the Beta-based
   attrition flags.
3. **Dimension foreign keys** - the hash keys that link each enrolment to its dimension rows.
4. **Derived attributes** - values deterministically derived from an already-drawn feature
   (e.g. an exact age within the drawn age band).

Finally, `FACT_EXPR` maps the generated column names onto the real fact-table schema. Splitting the
work into layers keeps each value produced at its natural point instead of relying on a later
"backfill" pass, and makes it explicit which columns are drawn vs derived vs deliberately null.

In [0]:
# Point Spark at the target catalog/schema, creating the schema if this is a fresh workspace.
spark.sql("USE CATALOG workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS student_aggregate")
spark.sql("USE SCHEMA student_aggregate")

# Fully-qualified names of every target table, kept in variables so the long names appear once.
FACT_FQN = "workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified"

# The target Delta table declares eftsl as decimal(10,0), which truncates fractional load to zero.
# Override to decimal(10,4) so band-sampled values (0.0437, 0.125, etc.) are preserved on write.
FACT_TYPE_OVERRIDES = {"eftsl": "decimal(10,4)"}
TP_DIM_FQN = "workspace.student_aggregate.dwh_learning_and_teaching__teaching_period"
COURSE_FQN = "workspace.student_aggregate.dwh_curriculum__course"
COURSE_OFFERING_FQN = "workspace.student_aggregate.dwh_curriculum__course_offering"
MODULE_FQN = "workspace.student_aggregate.dwh_curriculum__module"
MODULE_OFFERING_FQN = "workspace.student_aggregate.dwh_curriculum__module_offering"
UNIT_FQN = "workspace.student_aggregate.dwh_curriculum__unit"
UNIT_OFFERING_FQN = "workspace.student_aggregate.dwh_curriculum__unit_offering"
THESIS_FQN = "workspace.student_aggregate.dwh_curriculum__thesis"
STUDY_AREA_A_FQN = "workspace.student_aggregate.dwh_curriculum__study_area_a"
STUDY_AREA_B_FQN = "workspace.student_aggregate.dwh_curriculum__study_area_b"
ORG_DIM_FQN = "workspace.student_aggregate.dwh_internal_organisation__mapped_academic_organisation_hierarchy"

# Row count for each dimension table. Adjusted for realistic row count distributions keeps every table realistically representative and gives the fact-table
# foreign keys a large pool of dimension rows to point at.

# Synthetic dataset row counts, from current row count screenshot halved for computational constraints in synthetic data generation
FACT_N = 973_770

COURSE_N = 1_544
COURSE_OFFERING_N = 30_516

UNIT_N = 25_733
UNIT_OFFERING_N = 161_916

MODULE_N = 5
MODULE_OFFERING_N = 10
THESIS_N = 34

STUDY_AREA_A_N = 1_222
STUDY_AREA_B_N = 1_100

ORG_N = 822
TP_N = 3_083

# Unweighted field-of-education lookup lists (narrow/detailed labels are chosen per broad FoE below).
NARROW_PRIMARY_FOE_VALUES = get_lookup_values("narrow_primary_field_of_education")
DETAILED_PRIMARY_FOE_VALUES = get_lookup_values("detailed_primary_field_of_education")

# ---- Synthetic key helpers ----
# Referential integrity trick: every dimension row's key is deterministically derived from its 1-based
# index, and every fact row derives the SAME key from its id. Because both sides use the same formula,
# the fact-to-dimension joins in relationships.tmdl all resolve without an actual join at generation
# time.

# Zero-padding width for synthetic keys. Spark's lpad TRUNCATES an input longer than the requested
# length rather than leaving it intact, so a width smaller than the largest dimension index silently
# collapses distinct indexes onto the same key (index 10000 -> "1000", colliding with index 1000).
# A width of 4 previously capped every table at 9,999 distinct keys. The width must always exceed the
# digit count of the largest row count configured below; 12 leaves ample headroom.
KEY_WIDTH = 12

_LARGEST_DIMENSION_ROWS = max(
    COURSE_N, COURSE_OFFERING_N, UNIT_N, UNIT_OFFERING_N, MODULE_N, MODULE_OFFERING_N,
    THESIS_N, STUDY_AREA_A_N, STUDY_AREA_B_N, ORG_N, TP_N,
)
assert KEY_WIDTH >= len(str(_LARGEST_DIMENSION_ROWS)), (
    f"KEY_WIDTH={KEY_WIDTH} is too narrow for the largest dimension ({_LARGEST_DIMENSION_ROWS} rows). "
    "lpad would truncate and produce colliding keys."
)


# Build a (key, key_hash) pair from a prefix and an index column. The key is a readable synthetic
# string like "syn|course|000000000042"; the hash is its SHA-256, matching how the real tables store
# hash keys.
def keyed(prefix, idx_col):
    key = F.concat(F.lit(f"syn|{prefix}|"), F.lpad(idx_col.cast("string"), KEY_WIDTH, "0"))
    return key, F.sha2(key, 256)


# Same as keyed(), but derives the index from the fact row's id: pmod(id, size) + 1 maps every id into
# the valid 1..size range of a dimension so the foreign key always points at a real dimension row.
def keyed_from_id(prefix, size):
    idx = (F.pmod(F.col("id").cast("long"), F.lit(size)) + F.lit(1)).cast("int")
    return keyed(prefix, idx)


# The 1-based row index (id + 1). dbldatagen ids start at 0; dimension indexes start at 1, hence the + 1.
def row_idx():
    return F.col("id").cast("long") + F.lit(1)


# ---- Teaching-period dimension seed ----
# Generate TP_N teaching-period records independently. Each record receives a fixed calendar year and
# teaching period. Fact rows are assigned to one of these records through teaching_period_idx, matching
# the many-to-one pattern used by course, unit, and the other dimensions.
tp_spec = dg.DataGenerator(spark, rows=TP_N, partitions=4).withIdOutput()
tp_spec = add_weighted_column(tp_spec, "teaching_period", "teaching_period", "string")
tp_spec = add_weighted_column(tp_spec, "calendar_year", "calendar_year", "int")
tp_seed = (
    tp_spec.build()
    .withColumn("idx", F.col("id").cast("long") + F.lit(1))
    .drop("id")
)

# The dbldatagen spec: FACT_N size rows across 16 partitions. withIdOutput() exposes the built-in sequential
# "id" column (0..FACT_N), which is relied upon throughout as each row's stable identity - it seeds every
# deterministic draw and derives every foreign key. Each add_weighted_column attaches one V1 weighted
# categorical straight from distribution_values.csv.
# calendar_year and teaching_period are omitted here because they come from tp_seed via teaching_period_idx.
spec = dg.DataGenerator(
    spark,
    rows=FACT_N,
    partitions=16
).withIdOutput()

spec = add_weighted_column(spec, "socioeconomic_status", "socioeconomic_status", "string")
spec = add_weighted_column(spec, "regional_remote_status", "regional_remote_status", "string")
spec = add_weighted_column(spec, "course_level", "course_level", "string")
spec = add_weighted_column(spec, "course_group", "course_group", "string")
spec = add_weighted_column(spec, "broad_field_of_education", "broad_field_of_education", "string")
spec = add_weighted_column(spec, "gender", "gender", "string")
spec = add_weighted_column(spec, "age_band", "age_band", "string")
spec = add_weighted_column(spec, "international_domestic_student", "international_domestic_student", "string")
spec = add_weighted_column(spec, "indigenous_status", "indigenous_status", "string")
spec = add_weighted_column(spec, "attendance_mode_group", "attendance_mode_group", "string")

# Materialise the base rows (id + all the V1 weighted categoricals) into a DataFrame for extension.
df = spec.build()

# Assign each fact row to one teaching-period dimension record and join the matching year and period.
df = df.withColumn(
    "teaching_period_idx",
    (F.pmod(F.col("id").cast("long"), F.lit(TP_N)) + F.lit(1)).cast("long"),
)
df = df.join(
    tp_seed.select(
        F.col("idx").alias("teaching_period_idx"),
        "teaching_period",
        "calendar_year",
    ),
    on="teaching_period_idx",
    how="left",
)


# ---- Layer 1: identity, keys, and base attributes ----
# Values that follow directly from the id or a single drawn feature: de-identified hashes, the
# teaching-period key/hash, the enrolment year, and the international-student flag.

_tp_key, _tp_key_hash = keyed("tp", F.col("teaching_period_idx"))

df = (
    df.withColumn(
        # De-identified student hash: a stable SHA-256 of the id. Deterministic, so the same synthetic
        # student always maps to the same hash, but the id itself is not recoverable.
        "student_deidentified_hash",
        F.sha2(F.concat(F.lit("student_"), F.col("id").cast("string")), 256),
    )
    .withColumn(
        "enrolment_deidentified_hash",
        F.sha2(F.concat(F.lit("enrolment_"), F.col("id").cast("string")), 256),
    )
    .withColumn("teaching_period_key", _tp_key)
    .withColumn("teaching_period_key_hash", _tp_key_hash)
    .withColumn("enrolment_year", F.col("calendar_year").cast("int"))
    .withColumn(
        # Convenience boolean derived from the drawn domestic/international category.
        "is_international_student",
        (F.col("international_domestic_student") == F.lit("International")).cast("boolean"),
    )
)

# ==== Layer 2: conditional distribution features (V2 + attrition) ====

# ---- Student stage: commencing vs continuing ----
# ONE real feature (commencing_continuing) is drawn conditioned on field of education, plus a
# teaching-period-grained variant. Everything else here is DERIVED from those, not drawn independently,
# so all the "commencing" views of a row stay consistent with each other.
df = df.withColumn(
    "commencing_continuing",
    make_categorical_udf("cc_foe", "cc_foe")(F.col("id"), F.col("broad_field_of_education")),
)
df = df.withColumn(
    # Period-grain variant; coalesce falls back to the calendar-year value if the period has no data.
    "commencing_continuing_period",
    F.coalesce(
        make_categorical_udf("cc_tp", "cc_tp")(F.col("id"), F.col("teaching_period")),
        F.col("commencing_continuing"),
    ),
)
df = (
    # The 12-month and half-year views mirror the base value (there being no separate distribution for
    # them), and every is_commencing_* boolean is just "does the matching text value equal 'Commencing'".
    df.withColumn("commencing_continuing_12m", F.col("commencing_continuing"))
    .withColumn("commencing_continuing_half_year", F.col("commencing_continuing"))
    .withColumn("is_commencing", (F.col("commencing_continuing") == F.lit("Commencing")))
    .withColumn("is_commencing_12m", (F.col("commencing_continuing_12m") == F.lit("Commencing")))
    .withColumn("is_commencing_period", (F.col("commencing_continuing_period") == F.lit("Commencing")))
    .withColumn("is_commencing_half_year", (F.col("commencing_continuing_half_year") == F.lit("Commencing")))
)

# ---- Enrolment load (EFTSL) ----
# EFTSL = Equivalent Full-Time Student Load (1.0 = a full-time year). Its spread depends on both field
# of education and teaching period, so a band is picked using the precise (field, period) index, with a
# fallback to the field-only index if that pair is missing, and a decimal is then sampled within the band.
df = df.withColumn(
    "eftsl",
    make_banded_numeric_udf(
        "eftsl", "eftsl", "eftsl", as_int=False, fallback_bc_key="eftsl_foe", default_lo=0.0, default_hi=0.125
    )(F.col("id"), F.col("broad_field_of_education"), F.col("teaching_period")),
)

# Course admission load category (glossary: FT/PT at course level). Drawn from in-house att_type Beta
# parameters (Full-time vs Part-time), not from unit-grain EFTSL bands.
df = df.withColumn(
    "course_admission_load_category",
    make_load_category_udf(LOAD_CATEGORY_FT_PARAMS, LOAD_CATEGORY_PT_PARAMS, "load_category")(
        F.col("id")
    ),
)

# ---- Academic outcomes: cumulative credit points ----
# Passed / failed / withdrawn / enrolled are each their own distribution conditioned on field of
# education. After they are drawn, a real-world invariant is enforced with F.greatest: enrolled load must
# be at least passed + failed + withdrawn (enrolled can be higher because it also covers in-progress
# study). This keeps every generated row internally consistent.

df = (
    df.withColumn(
        "cumulative_credit_points_passed",
        make_banded_numeric_udf("cp_passed", "credit_point", "cp_passed", as_int=True)(
            F.col("id"), F.col("broad_field_of_education")
        ),
    )
    .withColumn(
        "cumulative_credit_points_failed",
        make_banded_numeric_udf("cp_failed", "credit_point", "cp_failed", as_int=True)(
            F.col("id"), F.col("broad_field_of_education")
        ),
    )
    .withColumn(
        "cumulative_credit_points_withdrawn",
        make_banded_numeric_udf("cp_withdrawn", "credit_point", "cp_withdrawn", as_int=True)(
            F.col("id"), F.col("broad_field_of_education")
        ),
    )
    .withColumn(
        "cumulative_credit_points_enrolled",
        make_banded_numeric_udf("cp_enrolled", "credit_point", "cp_enrolled", as_int=True)(
            F.col("id"), F.col("broad_field_of_education")
        ),
    )
    .withColumn(
        "cumulative_credit_points_enrolled",
        F.greatest(
            F.col("cumulative_credit_points_enrolled"),
            F.col("cumulative_credit_points_passed")
            + F.col("cumulative_credit_points_failed")
            + F.col("cumulative_credit_points_withdrawn"),
        ),
    )
    # Attrition outcomes drawn from the in-house Beta parameters, conditioned on field of education.
    # Course-level attrition (did not re-enrol in THIS course) is the primary Beta draw.
    .withColumn(
        "is_twelve_month_course_attrition",
        make_attrition_udf(ATTRITION_BETA_BY_FOE, ATTRITION_BETA_FALLBACK, "attrition_course")(
            F.col("id"), F.col("broad_field_of_education")
        ),
    )
    # Student-level attrition (did not re-enrol in ANY course) logically implies course attrition, so
    # it must be a subset: true only where course attrition is true, AND an independent Beta draw also
    # fires. This enforces student => course and keeps institution-level attrition rarer than
    # course-level attrition. (No student-specific rate was supplied, so the same field-of-education
    # Beta is reused as the subset selector.)
    .withColumn(
        "is_twelve_month_student_attrition",
        F.col("is_twelve_month_course_attrition")
        & make_attrition_udf(ATTRITION_BETA_BY_FOE, ATTRITION_BETA_FALLBACK, "attrition_student")(
            F.col("id"), F.col("broad_field_of_education")
        ),
    )
)

# ==== Layer 3: dimension foreign keys ====
# Each enrolment must link to real dimension rows (course, unit, study area, org, ...). Every
# foreign key is built with keyed_from_id, the same prefix + pmod(id, N) + 1 scheme the dimension tables use, so
# the keys are guaranteed to match existing dimension rows and every relationship in relationships.tmdl
# resolves. Each key comes as a readable string plus its SHA-256 hash (the real tables join on hashes).

_course_key, _course_key_hash = keyed_from_id("course", COURSE_N)
_course_offering_key, _course_offering_key_hash = keyed_from_id("course_offering", COURSE_OFFERING_N)
_curriculum_item_key, _curriculum_item_key_hash = keyed_from_id("unit", UNIT_N)
_curriculum_item_offering_key, _curriculum_item_offering_key_hash = keyed_from_id("unit_offering", UNIT_OFFERING_N)
_study_area_a_key, _study_area_a_key_hash = keyed_from_id("study_area_a", STUDY_AREA_A_N)
_study_area_b_key, _study_area_b_key_hash = keyed_from_id("study_area_b", STUDY_AREA_B_N)
_org_key, _org_key_hash = keyed_from_id("org_hier", ORG_N)

df = (
    df.withColumn("course_key", _course_key)
    .withColumn("course_key_hash", _course_key_hash)
    .withColumn("course_offering_key", _course_offering_key)
    .withColumn("course_offering_key_hash", _course_offering_key_hash)
    .withColumn("curriculum_item_key", _curriculum_item_key)
    .withColumn("curriculum_item_key_hash", _curriculum_item_key_hash)
    .withColumn("curriculum_item_offering_key", _curriculum_item_offering_key)
    .withColumn("curriculum_item_offering_key_hash", _curriculum_item_offering_key_hash)
    .withColumn("study_area_a_key", _study_area_a_key)
    .withColumn("study_area_a_key_hash", _study_area_a_key_hash)
    .withColumn("study_area_b_key", _study_area_b_key)
    .withColumn("study_area_b_key_hash", _study_area_b_key_hash)
    .withColumn("academic_organisation_hierarchy_key", _org_key)
    .withColumn("academic_organisation_hierarchy_key_hash", _org_key_hash)
)

# ==== Layer 4: derived attributes ====
# Everything here is DERIVED from an already-drawn feature or a defensible grain assumption - nothing is
# invented. Codes/labels that cannot be grounded are deliberately not fabricated (those stay NULL).

# Grain assumption: every generated row represents an actual enrolment that carries study load, so both
# flags are always true for this fact.
df = df.withColumn("is_enrolment", F.lit(True)).withColumn("is_study_load", F.lit(True))

# age_at_census: the drawn feature is only an age BAND (e.g. "20 to 24"); the schema requires an exact age.
# Each band is mapped to its [lower, upper] bounds, and a deterministic age is then picked inside that
# range using pmod(id, range_size). This "disaggregates" the band into a concrete age consistent with it.
_age_lower = (
    F.when(F.col("age_band") == F.lit("14 or less"), F.lit(13))
    .when(F.col("age_band") == F.lit("15 to 19"), F.lit(15))
    .when(F.col("age_band") == F.lit("20 to 24"), F.lit(20))
    .when(F.col("age_band") == F.lit("25 to 29"), F.lit(25))
    .when(F.col("age_band") == F.lit("30 to 34"), F.lit(30))
    .when(F.col("age_band") == F.lit("35 to 39"), F.lit(35))
    .when(F.col("age_band") == F.lit("40 to 44"), F.lit(40))
    .when(F.col("age_band") == F.lit("45 to 49"), F.lit(45))
    .when(F.col("age_band") == F.lit("50 to 54"), F.lit(50))
    .when(F.col("age_band") == F.lit("55 to 59"), F.lit(55))
    .when(F.col("age_band") == F.lit("60 or more"), F.lit(60))
    .otherwise(F.lit(20))
)
_age_upper = (
    F.when(F.col("age_band") == F.lit("14 or less"), F.lit(14))
    .when(F.col("age_band") == F.lit("15 to 19"), F.lit(19))
    .when(F.col("age_band") == F.lit("20 to 24"), F.lit(24))
    .when(F.col("age_band") == F.lit("25 to 29"), F.lit(29))
    .when(F.col("age_band") == F.lit("30 to 34"), F.lit(34))
    .when(F.col("age_band") == F.lit("35 to 39"), F.lit(39))
    .when(F.col("age_band") == F.lit("40 to 44"), F.lit(44))
    .when(F.col("age_band") == F.lit("45 to 49"), F.lit(49))
    .when(F.col("age_band") == F.lit("50 to 54"), F.lit(54))
    .when(F.col("age_band") == F.lit("55 to 59"), F.lit(59))
    .when(F.col("age_band") == F.lit("60 or more"), F.lit(70))
    .otherwise(F.lit(24))
)
df = df.withColumn(
    "age_at_census",
    (_age_lower + F.pmod(F.col("id").cast("int"), (_age_upper - _age_lower + F.lit(1)))).cast("int"),
)

# Census dates: the census date is the official enrolment snapshot date, which depends on the teaching
# period. Each period is mapped to its standard census date within the row's calendar year (Semester 1 ~
# late March, Semester 2 ~ late August, Summer ~ mid January, otherwise a mid-April default).
_census_date = (
    F.when(F.col("teaching_period").contains("Semester 1"), F.make_date(F.col("calendar_year"), F.lit(3), F.lit(31)))
    .when(F.col("teaching_period").contains("Semester 2"), F.make_date(F.col("calendar_year"), F.lit(8), F.lit(31)))
    .when(F.col("teaching_period").contains("Summer"), F.make_date(F.col("calendar_year"), F.lit(1), F.lit(15)))
    .otherwise(F.make_date(F.col("calendar_year"), F.lit(4), F.lit(15)))
)
df = df.withColumn("census_date", _census_date).withColumn(
    "course_enrolment_census_date", F.col("census_date")
)

# FACT_EXPR is the mapping from the real fact-table column names to the expressions that fill them.
# align_to_target uses this dict: any fact column listed here gets the generated value; any fact column
# NOT listed is written as a typed NULL. This is the single place that decides what each fact column contains, and
# it makes the "drawn vs derived vs deliberately null" decision explicit and auditable.
FACT_EXPR = {
    "enrolment_deidentified_hash": F.col("enrolment_deidentified_hash"),
    "student_deidentified_hash": F.col("student_deidentified_hash"),
    "socioeconomic_status": F.col("socioeconomic_status"),
    "regional_remote_status": F.col("regional_remote_status"),
    "age_band": F.col("age_band"),
    "is_international_student": F.col("is_international_student"),
    "student_gender": F.col("gender"),
    "attendance_mode": F.col("attendance_mode_group"),
    # Student_is_first_nations_student confirms both indigenous status and first nations
    "student_is_first_nations_student": (F.col("indigenous_status") == F.lit("First Nations")).cast(
        "boolean"
    ),
    "student_is_international_student": F.col("is_international_student"),
    "international_domestic_enrolment": F.col("international_domestic_student"),
    "international_domestic_student": F.col("international_domestic_student"),
    "enrolment_year": F.col("enrolment_year"),
    "eftsl": F.col("eftsl"),
    "course_admission_load_category": F.col("course_admission_load_category"),
    "teaching_period_key": F.col("teaching_period_key"),
    "teaching_period_key_hash": F.col("teaching_period_key_hash"),
    # V2: student stage (commencing/continuing) and derived flags
    "commencing_continuing": F.col("commencing_continuing"),
    "commencing_continuing_12m": F.col("commencing_continuing_12m"),
    "commencing_continuing_period": F.col("commencing_continuing_period"),
    "commencing_continuing_half_year": F.col("commencing_continuing_half_year"),
    "is_commencing": F.col("is_commencing"),
    "is_commencing_12m": F.col("is_commencing_12m"),
    "is_commencing_period": F.col("is_commencing_period"),
    "is_commencing_half_year": F.col("is_commencing_half_year"),
    # V2: cumulative credit-point outcomes
    "cumulative_credit_points_passed": F.col("cumulative_credit_points_passed"),
    "cumulative_credit_points_enrolled": F.col("cumulative_credit_points_enrolled"),
    "cumulative_credit_points_failed": F.col("cumulative_credit_points_failed"),
    "cumulative_credit_points_withdrawn": F.col("cumulative_credit_points_withdrawn"),
    # In-house Beta distribution: attrition outcomes
    "is_twelve_month_course_attrition": F.col("is_twelve_month_course_attrition"),
    "is_twelve_month_student_attrition": F.col("is_twelve_month_student_attrition"),
    # V2: foreign keys for referential integrity (relationships.tmdl)
    "course_key": F.col("course_key"),
    "course_key_hash": F.col("course_key_hash"),
    "course_offering_key": F.col("course_offering_key"),
    "course_offering_key_hash": F.col("course_offering_key_hash"),
    "curriculum_item_key": F.col("curriculum_item_key"),
    "curriculum_item_key_hash": F.col("curriculum_item_key_hash"),
    "curriculum_item_offering_key": F.col("curriculum_item_offering_key"),
    "curriculum_item_offering_key_hash": F.col("curriculum_item_offering_key_hash"),
    "study_area_a_key": F.col("study_area_a_key"),
    "study_area_a_key_hash": F.col("study_area_a_key_hash"),
    "study_area_b_key": F.col("study_area_b_key"),
    "study_area_b_key_hash": F.col("study_area_b_key_hash"),
    "academic_organisation_hierarchy_key": F.col("academic_organisation_hierarchy_key"),
    "academic_organisation_hierarchy_key_hash": F.col("academic_organisation_hierarchy_key_hash"),
    # Derived attributes (disaggregation / grounded grain assumptions)
    "age_at_census": F.col("age_at_census"),
    "is_enrolment": F.col("is_enrolment"),
    "is_study_load": F.col("is_study_load"),
    "census_date": F.col("census_date"),
    "course_enrolment_census_date": F.col("course_enrolment_census_date"),
}

## Write the dimension tables and the fact table

Each dimension table (course, unit, study area, org, teaching period, ...) is built, aligned
to its schema, and written to Delta. The dimension keys use the same `keyed()` scheme as the
fact-table foreign keys, so the two sides line up. Every write uses overwrite mode, so re-running
regenerates the whole dataset from scratch.

In [0]:
# base_dim builds the skeleton of a dimension table: idx = 1..n plus the matching (key, key_hash). This
# is the counterpart to keyed_from_id on the fact side - same prefix + index, so the keys match.
def base_dim(prefix, n):
    x = spark.range(1, n + 1).withColumnRenamed("id", "idx")
    key, key_hash = keyed(prefix, F.col("idx"))
    return x.withColumn("key", key).withColumn("key_hash", key_hash)


# Map each narrow FoE label to its broad parent so narrow/detailed align with broad_primary_field_of_education.
_NARROW_TO_BROAD = {
    "Business and Management": "MANAGEMENT AND COMMERCE",
    "Other Engineering and Related Technologies": "ENGINEERING AND RELATED TECHNOLOGIES",
    "Teacher Education": "EDUCATION",
    "Nursing": "HEALTH",
    "Other Information Technology": "INFORMATION TECHNOLOGY",
    "Behavioural Science": "SOCIETY AND CULTURE",
    "INFORMATION TECHNOLOGY, n.f.d.": "INFORMATION TECHNOLOGY",
    "Architecture and Urban Environment": "ARCHITECTURE AND BUILDING",
    "Law": "MANAGEMENT AND COMMERCE",
    "MANAGEMENT AND COMMERCE, n.f.d.": "MANAGEMENT AND COMMERCE",
    "Other Natural and Physical Sciences": "NATURAL AND PHYSICAL SCIENCES",
    "Not applicable": "NONE",
    "Other Management and Commerce": "MANAGEMENT AND COMMERCE",
    "Other Health": "HEALTH",
    "Building": "ARCHITECTURE AND BUILDING",
    "Communication and Media Studies": "SOCIETY AND CULTURE",
    "Other Education": "EDUCATION",
    "Visual Arts and Crafts": "CREATIVE ARTS",
    "Human Welfare Studies and Services": "SOCIETY AND CULTURE",
    "Justice and Law Enforcement": "SOCIETY AND CULTURE",
    "Public Health": "HEALTH",
    "Graphic and Design Studies": "CREATIVE ARTS",
    "Radiography": "HEALTH",
    "Other Creative Arts": "CREATIVE ARTS",
    "CREATIVE ARTS, n.f.d.": "CREATIVE ARTS",
    "Pharmacy": "HEALTH",
    "Optical Science": "HEALTH",
    "HEALTH, n.f.d.": "HEALTH",
    "Language and Literature": "SOCIETY AND CULTURE",
    "Rehabilitation Therapies": "HEALTH",
    "Mathematical Sciences": "NATURAL AND PHYSICAL SCIENCES",
    "Biological Sciences": "NATURAL AND PHYSICAL SCIENCES",
    "Electrical and Electronic Engineering and Technology": "ENGINEERING AND RELATED TECHNOLOGIES",
    "Political Science and Policy Studies": "SOCIETY AND CULTURE",
    "Computer Science": "INFORMATION TECHNOLOGY",
    "Performing Arts": "CREATIVE ARTS",
    "Civil Engineering": "ENGINEERING AND RELATED TECHNOLOGIES",
    "Mechanical and Industrial Engineering and Technology": "ENGINEERING AND RELATED TECHNOLOGIES",
    "Manufacturing Engineering and Technology": "ENGINEERING AND RELATED TECHNOLOGIES",
}

BROAD_FOE_VALUES = get_weighted_distribution("broad_field_of_education")[0]
NARROW_BY_BROAD = {broad: [] for broad in BROAD_FOE_VALUES}
for _narrow in NARROW_PRIMARY_FOE_VALUES:
    _broad = _NARROW_TO_BROAD.get(_narrow, "SOCIETY AND CULTURE")
    NARROW_BY_BROAD.setdefault(_broad, []).append(_narrow)
for _broad, _labels in list(NARROW_BY_BROAD.items()):
    if not _labels:
        NARROW_BY_BROAD[_broad] = ["Not applicable"]


def _foe_label_stem(label):
    return label.split(":")[0].split(",")[0].strip().upper()


DETAILED_BY_NARROW = {}
for _narrow in NARROW_PRIMARY_FOE_VALUES:
    _stem = _foe_label_stem(_narrow)
    _matches = [
        _detailed
        for _detailed in DETAILED_PRIMARY_FOE_VALUES
        if (
            _stem == "NOT APPLICABLE"
            and _detailed == "Not applicable"
        )
        or (
            _stem != "NOT APPLICABLE"
            and (
                _stem in _foe_label_stem(_detailed)
                or _foe_label_stem(_detailed).startswith(_stem)
            )
        )
    ]
    DETAILED_BY_NARROW[_narrow] = _matches or DETAILED_PRIMARY_FOE_VALUES


@F.udf(StringType())
def pick_narrow_foe_for_broad(broad, idx):
    candidates = NARROW_BY_BROAD.get(broad) or NARROW_BY_BROAD.get("NONE") or NARROW_PRIMARY_FOE_VALUES
    return candidates[(int(idx) - 1) % len(candidates)]


@F.udf(StringType())
def pick_detailed_foe_for_narrow(narrow, idx):
    candidates = DETAILED_BY_NARROW.get(narrow) or DETAILED_PRIMARY_FOE_VALUES
    return candidates[(int(idx) - 1) % len(candidates)]


# Shared FoE column maps: course uses *_primary_* schema names; other curriculum tables use the
# non-primary names but the same dim_seed values so all six tables stay aligned at a given idx.
COURSE_PRIMARY_FOE_COL_MAP = {
    "broad_primary_field_of_education": F.col("broad_primary_field_of_education"),
    "broad_primary_field_of_education_code": F.col("broad_field_of_education_code"),
    "narrow_primary_field_of_education": F.col("narrow_primary_field_of_education"),
    "narrow_primary_field_of_education_code": F.col("narrow_field_of_education_code"),
    "detailed_primary_field_of_education": F.col("detailed_primary_field_of_education"),
    "detailed_primary_field_of_education_code": F.col("detailed_field_of_education_code"),
    "is_primary_field_of_education_non_traditional_area_for_women": F.lit(None),
}

CURRICULUM_FOE_COL_MAP = {
    "broad_field_of_education_code": F.col("broad_field_of_education_code"),
    "broad_field_of_education": F.col("broad_primary_field_of_education"),
    "narrow_field_of_education_code": F.col("narrow_field_of_education_code"),
    "narrow_field_of_education": F.col("narrow_primary_field_of_education"),
    "detailed_field_of_education_code": F.col("detailed_field_of_education_code"),
    "detailed_field_of_education": F.col("detailed_primary_field_of_education"),
    "is_field_of_education_non_traditional_area_for_women": F.lit(None),
}

MAX_CURRICULUM_DIM_IDX = max(
    COURSE_N, UNIT_N, MODULE_N, THESIS_N, STUDY_AREA_A_N, STUDY_AREA_B_N
)

# dim_seed carries the attributes SHARED across several dimension tables (course group, field of
# education, ...). They are computed once from the fact rows and joined in by idx, so, e.g., the course
# and unit dimensions describe the same field of education for a given idx. Building once avoids
# recomputing the same logic in every dimension below.
dim_seed = (
    df.select(
        row_idx().cast("int").alias("idx"),  # 1-based index; joins to each dimension's idx
        F.col("course_group"),
        F.col("broad_field_of_education").alias("broad_primary_field_of_education"),
        F.col("broad_field_of_education"),
    )
    .filter(F.col("idx") <= F.lit(MAX_CURRICULUM_DIM_IDX))
    .withColumn(
        "course_group_code",
        F.when(F.col("course_group") == F.lit("HDR"), F.lit("HDR"))
        .when(F.col("course_group") == F.lit("Other"), F.lit("OTH"))
        .when(F.col("course_group") == F.lit("Postgraduate (Coursework)"), F.lit("PG"))
        .otherwise(F.lit("UG")),
    )
    # Narrow/detailed labels are chosen from lookup lists conditioned on broad FoE (not independent idx cycling).
    .withColumn(
        "narrow_primary_field_of_education",
        pick_narrow_foe_for_broad(F.col("broad_primary_field_of_education"), F.col("idx")),
    )
    .withColumn(
        "detailed_primary_field_of_education",
        pick_detailed_foe_for_narrow(F.col("narrow_primary_field_of_education"), F.col("idx")),
    )
    # The official field-of-education *codes* (ASCED codes) were not supplied, so they are left NULL
    # rather than invented as codes that could be mistaken for the real classification.
    .withColumn("broad_field_of_education_code", F.lit(None))
    .withColumn("narrow_field_of_education_code", F.lit(None))
    .withColumn("detailed_field_of_education_code", F.lit(None))
    # The non-primary field-of-education columns mirror the primary ones (a single field is modelled).
    .withColumn("narrow_field_of_education", F.col("narrow_primary_field_of_education"))
    .withColumn("detailed_field_of_education", F.col("detailed_primary_field_of_education"))
    .withColumn("is_primary_field_of_education_non_traditional_area_for_women", F.lit(False))
    .withColumn("is_field_of_education_non_traditional_area_for_women", F.lit(False))
)


"""
Does the following for student_aggregate.dwh_curriculum__course:
- Builds base rows for the dimensional table using base_dim
- Joins dim_seed onto the dimension table (shared attributes e.g. course group)
- align_to_target maps prepared columns into the specific schema/table/columns
- saveAsTable writes the synthetic generated rows to the Delta table
"""
course_dim = base_dim("course", COURSE_N).join(dim_seed, on="idx", how="left")
df_course = align_to_target(
    course_dim,
    COURSE_FQN,
    {
        "course_key": F.col("key"),
        "course_key_hash": F.col("key_hash"),
        "course_code": F.lit(None),
        "course_version_number": F.lit(None),
        "title": F.lit(None),
        "course_code_and_title": F.lit(None),
        "short_title": F.lit(None),
        "abbreviated_title": F.lit(None),
        "course_level": F.col("course_group"),
        "course_level_code": F.lit(None),
        "course_type_code": F.lit(None),
        "course_type": F.lit(None),
        "course_group_code": F.lit(None),
        "course_group": F.col("course_group"),
        **COURSE_PRIMARY_FOE_COL_MAP,
    },
)
df_course.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(COURSE_FQN)

"""
Does the following for student_aggregate.dwh_curriculum__course_offering:
- Builds base rows for the dimensional table using base_dim
- Joins dim_seed onto the dimension table (shared attributes e.g. course group)
- align_to_target maps prepared columns into the specific schema/table/columns
- saveAsTable writes the synthetic generated rows to the Delta table
"""
course_offering_dim = base_dim("course_offering", COURSE_OFFERING_N).withColumn(
    "course_idx", (F.pmod(F.col("idx") - 1, F.lit(COURSE_N)) + F.lit(1)).cast("int")
)
co_course_key, co_course_key_hash = keyed("course", F.col("course_idx"))
df_course_offering = align_to_target(
    course_offering_dim,
    COURSE_OFFERING_FQN,
    {
        "course_offering_key": F.col("key"),
        "course_offering_key_hash": F.col("key_hash"),
        "course_code": F.lit(None),
        "course_version_number": F.lit(None),
        "course_offering": F.lit(None),
        "course_key": co_course_key,
        "course_key_hash": co_course_key_hash,
    },
)
df_course_offering.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    COURSE_OFFERING_FQN
)

"""
Does the following for student_aggregate.dwh_curriculum__unit:
- Builds base rows for the dimensional table using base_dim
- Joins dim_seed onto the dimension table (shared attributes e.g. course group)
- align_to_target maps prepared columns into the specific schema/table/columns
- saveAsTable writes the synthetic generated rows to the Delta table
"""
unit_dim = base_dim("unit", UNIT_N).join(dim_seed, on="idx", how="left")
df_unit = align_to_target(
    unit_dim,
    UNIT_FQN,
    {
        "unit_key": F.col("key"),
        "unit_key_hash": F.col("key_hash"),
        **CURRICULUM_FOE_COL_MAP,
    },
)
df_unit.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(UNIT_FQN)

"""
Does the following for student_aggregate.dwh_curriculum__module:
- Builds base rows for the dimensional table using base_dim
- Joins dim_seed onto the dimension table (shared attributes e.g. course group)
- align_to_target maps prepared columns into the specific schema/table/columns
- saveAsTable writes the synthetic generated rows to the Delta table
"""
module_dim = base_dim("module", MODULE_N).join(dim_seed, on="idx", how="left")
df_module = align_to_target(
    module_dim,
    MODULE_FQN,
    {
        "module_key": F.col("key"),
        "module_key_hash": F.col("key_hash"),
        "module_code": F.lit(None),
        "title": F.lit(None),
        **CURRICULUM_FOE_COL_MAP,
    },
)
df_module.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(MODULE_FQN)

"""
Does the following for student_aggregate.dwh_curriculum__thesis:
- Builds base rows for the dimensional table using base_dim
- Joins dim_seed onto the dimension table (shared attributes e.g. course group)
- align_to_target maps prepared columns into the specific schema/table/columns
- saveAsTable writes the synthetic generated rows to the Delta table
"""
thesis_dim = base_dim("thesis", THESIS_N).join(dim_seed, on="idx", how="left")
df_thesis = align_to_target(
    thesis_dim,
    THESIS_FQN,
    {
        "thesis_key": F.col("key"),
        "thesis_key_hash": F.col("key_hash"),
        "thesis_code": F.lit(None),
        "title": F.lit(None),
        **CURRICULUM_FOE_COL_MAP,
    },
)
df_thesis.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(THESIS_FQN)

"""
Does the following for student_aggregate.dwh_curriculum__unit_offering:
- Builds base rows for the dimensional table using base_dim
- Joins dim_seed onto the dimension table (shared attributes e.g. course group)
- align_to_target maps prepared columns into the specific schema/table/columns
- saveAsTable writes the synthetic generated rows to the Delta table
"""
unit_offering_dim = base_dim("unit_offering", UNIT_OFFERING_N).withColumn(
    "unit_idx", (F.pmod(F.col("idx") - 1, F.lit(UNIT_N)) + F.lit(1)).cast("int")
)
uo_unit_key, uo_unit_key_hash = keyed("unit", F.col("unit_idx"))
df_unit_offering = align_to_target(
    unit_offering_dim,
    UNIT_OFFERING_FQN,
    {
        "unit_offering_key": F.col("key"),
        "unit_offering_key_hash": F.col("key_hash"),
        "unit_key": uo_unit_key,
        "unit_key_hash": uo_unit_key_hash,
    },
)
df_unit_offering.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    UNIT_OFFERING_FQN
)

"""
Does the following for student_aggregate.dwh_curriculum__module_offering:
- Builds base rows for the dimensional table using base_dim
- Joins dim_seed onto the dimension table (shared attributes e.g. course group)
- align_to_target maps prepared columns into the specific schema/table/columns
- saveAsTable writes the synthetic generated rows to the Delta table
"""
module_offering_dim = base_dim("module_offering", MODULE_OFFERING_N).withColumn(
    "module_idx", (F.pmod(F.col("idx") - 1, F.lit(MODULE_N)) + F.lit(1)).cast("int")
)
mo_module_key, mo_module_key_hash = keyed("module", F.col("module_idx"))
df_module_offering = align_to_target(
    module_offering_dim,
    MODULE_OFFERING_FQN,
    {
        "module_offering_key": F.col("key"),
        "module_offering_key_hash": F.col("key_hash"),
        "module_key": mo_module_key,
        "module_key_hash": mo_module_key_hash,
    },
)
df_module_offering.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    MODULE_OFFERING_FQN
)

"""
Does the following for student_aggregate.dwh_curriculum__study_area_a:
- Builds base rows for the dimensional table using base_dim
- Joins dim_seed onto the dimension table (shared attributes e.g. course group)
- align_to_target maps prepared columns into the specific schema/table/columns
- saveAsTable writes the synthetic generated rows to the Delta table
"""
study_area_a_dim = base_dim("study_area_a", STUDY_AREA_A_N).join(dim_seed, on="idx", how="left")
df_study_area_a = align_to_target(
    study_area_a_dim,
    STUDY_AREA_A_FQN,
    {
        "study_area_a_key": F.col("key"),
        "study_area_a_key_hash": F.col("key_hash"),
        **CURRICULUM_FOE_COL_MAP,
    },
)
df_study_area_a.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    STUDY_AREA_A_FQN
)

"""
Does the following for student_aggregate.dwh_curriculum__study_area_b:
- Builds base rows for the dimensional table using base_dim
- Joins dim_seed onto the dimension table (shared attributes e.g. course group)
- align_to_target maps prepared columns into the specific schema/table/columns
- saveAsTable writes the synthetic generated rows to the Delta table
"""
study_area_b_dim = base_dim("study_area_b", STUDY_AREA_B_N).join(dim_seed, on="idx", how="left")
df_study_area_b = align_to_target(
    study_area_b_dim,
    STUDY_AREA_B_FQN,
    {
        "study_area_b_key": F.col("key"),
        "study_area_b_key_hash": F.col("key_hash"),
        **CURRICULUM_FOE_COL_MAP,
    },
)
df_study_area_b.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    STUDY_AREA_B_FQN
)

"""
Does the following for student_aggregate.dwh_internal_organisation__mapped_academic_organisation_hierarchy:
- Builds base rows for the dimensional table using base_dim
- Joins dim_seed onto the dimension table (shared attributes e.g. course group)
- align_to_target maps prepared columns into the specific schema/table/columns
- saveAsTable writes the synthetic generated rows to the Delta table
"""
org_dim = base_dim("org_hier", ORG_N)
df_org = align_to_target(
    org_dim,
    ORG_DIM_FQN,
    {
        "mapped_academic_organisation_hierarchy_key": F.col("key"),
        "mapped_academic_organisation_hierarchy_key_hash": F.col("key_hash"),
        "academic_organisation_hierarchy_key": F.col("key"),
        "academic_organisation_hierarchy_key_hash": F.col("key_hash"),
    },
)
df_org.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(ORG_DIM_FQN)

# Build and write the FACT table itself: align the enriched df to the fact schema via FACT_EXPR (any
# unmapped fact column becomes a typed NULL), then overwrite the Delta table.
df_fact = align_to_target(df, FACT_FQN, FACT_EXPR, FACT_TYPE_OVERRIDES)
df_fact.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(FACT_FQN)

# Build the teaching-period dimension from tp_seed. Fact rows reference these records through
# teaching_period_idx using the same keyed("tp", idx) scheme as Layer 1.
_tp_dim_key, _tp_dim_key_hash = keyed("tp", F.col("idx"))
tp_dim = (
    tp_seed.withColumn(
        "start_date",
        F.date_add(
            F.make_date(F.col("calendar_year"), F.lit(1), F.lit(1)),
            F.pmod(F.col("idx"), F.lit(300)).cast("int"),
        ),
    )
    .withColumn(
        "end_date",
        F.date_add(
            F.col("start_date"),
            (F.lit(30) + F.pmod(F.col("idx"), F.lit(35))).cast("int"),
        ),
    )
    .withColumn("teaching_period_key", _tp_dim_key)
    .withColumn("teaching_period_key_hash", _tp_dim_key_hash)
    .withColumn("teaching_period_code", F.lit(None).cast("string"))
)

TP_EXPR = {
    "teaching_period": F.col("teaching_period"),
    "teaching_period_key": F.col("teaching_period_key"),
    "teaching_period_key_hash": F.col("teaching_period_key_hash"),
    "teaching_period_code": F.col("teaching_period_code"),
    "calendar_year": F.col("calendar_year"),
    "start_date": F.col("start_date"),
    "end_date": F.col("end_date"),
}

df_tp = align_to_target(tp_dim, TP_DIM_FQN, TP_EXPR)
df_tp.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(TP_DIM_FQN)

## Validation

Quick checks that the tables were written and look sensible: a random 20-row sample from each
table (to confirm by inspection that values vary according to the weightings), followed by a
row-count check confirming each table matches its configured row count.

In [0]:
%sql
-- Random 20 sample from student management fact table
SELECT * FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified ORDER BY rand() LIMIT 20;

In [0]:
%sql
-- Random 20 sample from curriculum course table
SELECT * FROM workspace.student_aggregate.dwh_curriculum__course ORDER BY rand() LIMIT 20;

In [0]:
%sql
-- Random 20 sample from curriculum course offering table
SELECT * FROM workspace.student_aggregate.dwh_curriculum__course_offering ORDER BY rand() LIMIT 20;

In [0]:
%sql
-- Random 20 sample from curriculum module table
SELECT * FROM workspace.student_aggregate.dwh_curriculum__module ORDER BY rand() LIMIT 20;


In [0]:
%sql
-- Random 20 sample from curriculum module offering table
SELECT * FROM workspace.student_aggregate.dwh_curriculum__module_offering ORDER BY rand() LIMIT 20;


In [0]:
%sql
-- Random 20 sample from curriculum unit table
SELECT * FROM workspace.student_aggregate.dwh_curriculum__unit ORDER BY rand() LIMIT 20;


In [0]:
%sql
-- Random 20 sample from curriculum unit offering table
SELECT * FROM workspace.student_aggregate.dwh_curriculum__unit_offering ORDER BY rand() LIMIT 20;


In [0]:
%sql
-- Random 20 sample from curriculum thesis table
SELECT * FROM workspace.student_aggregate.dwh_curriculum__thesis ORDER BY rand() LIMIT 20;


In [0]:
%sql
-- Random 20 sample from curriculum study area A table
SELECT * FROM workspace.student_aggregate.dwh_curriculum__study_area_a ORDER BY rand() LIMIT 20;


In [0]:
%sql
-- Random 20 sample from curriculum study area B table
SELECT * FROM workspace.student_aggregate.dwh_curriculum__study_area_b ORDER BY rand() LIMIT 20;


In [0]:
%sql
-- Random 20 sample from internal organisation hierarchy table
SELECT * FROM workspace.student_aggregate.dwh_internal_organisation__mapped_academic_organisation_hierarchy ORDER BY rand() LIMIT 20;

In [0]:
%sql
-- Random 20 sample from learning and teaching period table
SELECT * FROM workspace.student_aggregate.dwh_learning_and_teaching__teaching_period ORDER BY rand() LIMIT 20;

Confirm rows of synthetic data have been generated for each table

In [0]:
%sql
-- Select all tables, join them all and check table size to ensure rows generated for all tables
SELECT 'workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified' AS table_name, COUNT(*) AS row_count
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified
UNION ALL
SELECT 'workspace.student_aggregate.dwh_curriculum__course', COUNT(*)
FROM workspace.student_aggregate.dwh_curriculum__course
UNION ALL
SELECT 'workspace.student_aggregate.dwh_curriculum__course_offering', COUNT(*)
FROM workspace.student_aggregate.dwh_curriculum__course_offering
UNION ALL
SELECT 'workspace.student_aggregate.dwh_curriculum__module', COUNT(*)
FROM workspace.student_aggregate.dwh_curriculum__module
UNION ALL
SELECT 'workspace.student_aggregate.dwh_curriculum__module_offering', COUNT(*)
FROM workspace.student_aggregate.dwh_curriculum__module_offering
UNION ALL
SELECT 'workspace.student_aggregate.dwh_curriculum__unit', COUNT(*)
FROM workspace.student_aggregate.dwh_curriculum__unit
UNION ALL
SELECT 'workspace.student_aggregate.dwh_curriculum__unit_offering', COUNT(*)
FROM workspace.student_aggregate.dwh_curriculum__unit_offering
UNION ALL
SELECT 'workspace.student_aggregate.dwh_curriculum__thesis', COUNT(*)
FROM workspace.student_aggregate.dwh_curriculum__thesis
UNION ALL
SELECT 'workspace.student_aggregate.dwh_curriculum__study_area_a', COUNT(*)
FROM workspace.student_aggregate.dwh_curriculum__study_area_a
UNION ALL
SELECT 'workspace.student_aggregate.dwh_curriculum__study_area_b', COUNT(*)
FROM workspace.student_aggregate.dwh_curriculum__study_area_b
UNION ALL
SELECT 'workspace.student_aggregate.dwh_internal_organisation__mapped_academic_organisation_hierarchy', COUNT(*)
FROM workspace.student_aggregate.dwh_internal_organisation__mapped_academic_organisation_hierarchy
UNION ALL
SELECT 'workspace.student_aggregate.dwh_learning_and_teaching__teaching_period', COUNT(*)
FROM workspace.student_aggregate.dwh_learning_and_teaching__teaching_period
ORDER BY table_name;